# Amazon Beauty Dataset: Loading and Cleaning

This notebook loads the public Amazon Beauty ratings dataset, checks for missing and irrelevant values, cleans the data, and saves a prepared CSV for SASRec/FedSASRec experiments.

## Dataset source

Official UCSD source: https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/index.html

The ratings-only file contains user ID, item ID, rating, and timestamp. These are the fields needed for sequential recommendation.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

In [ ]:
# Public ratings-only file
DATA_URL = (
    'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/'
    'categoryFilesSmall/All_Beauty.csv'
)
LOCAL_FILE = Path('All_Beauty.csv')

# Use a local copy when available; otherwise load from the public URL.
if LOCAL_FILE.exists():
    df = pd.read_csv(LOCAL_FILE)
    print(f'Loaded local file: {LOCAL_FILE}')
else:
    df = pd.read_csv(DATA_URL)
    print('Loaded dataset from the public URL')

# Normalize common Amazon CSV header variations.
column_aliases = {
    'UserId': 'reviewerID',
    'user_id': 'reviewerID',
    'user': 'reviewerID',
    'ProductId': 'asin',
    'item_id': 'asin',
    'item': 'asin',
    'Rating': 'overall',
    'rating': 'overall',
    'Timestamp': 'unixReviewTime',
    'timestamp': 'unixReviewTime'
}
df = df.rename(columns=column_aliases)

print('Shape:', df.shape)
df.head()

In [ ]:
# Inspect column names and data types
print('Columns:', list(df.columns))
print('\nData types:')
print(df.dtypes)

print('\nDataset dimensions:')
print(f'Rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')

## Check null and empty values

Missing user IDs, item IDs, ratings, or timestamps cannot be used to create reliable sequences.

In [ ]:
# Count missing values in every column
null_counts = df.isna().sum().sort_values(ascending=False)
print(null_counts)

# Count empty strings in text-like columns
empty_counts = {}
for column in df.select_dtypes(include='object').columns:
    empty_counts[column] = df[column].astype('string').str.strip().eq('').sum()

print('\nEmpty-string counts:')
print(pd.Series(empty_counts).sort_values(ascending=False))

In [ ]:
# Check duplicated rows and duplicated user-item interactions
print('Exact duplicate rows:', df.duplicated().sum())

id_columns = [column for column in ['reviewerID', 'asin'] if column in df.columns]
if len(id_columns) == 2:
    print('Repeated user-item pairs:', df.duplicated(subset=id_columns).sum())

# Show unique-value counts to identify constant or irrelevant columns
unique_counts = df.nunique(dropna=False).sort_values()
print('\nUnique values per column:')
print(unique_counts)

## Select relevant columns

For the initial SASRec/FedSASRec experiment, we only need the user, item, rating, and timestamp. Review text and other fields are not required for the sequence model.

In [ ]:
required_columns = ['reviewerID', 'asin', 'overall', 'unixReviewTime']
missing_required = [column for column in required_columns if column not in df.columns]

if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

clean_df = df[required_columns].copy()
clean_df = clean_df.rename(columns={
    'reviewerID': 'user_id',
    'asin': 'item_id',
    'overall': 'rating',
    'unixReviewTime': 'timestamp'
})
clean_df.head()

## Clean invalid and irrelevant records

The following operations remove incomplete identifiers, invalid ratings, invalid timestamps, exact duplicates, and repeated user-item rows. Keeping the earliest timestamp for a repeated user-item pair creates one chronological interaction per pair for this simple baseline.

In [ ]:
before_rows = len(clean_df)

# Remove missing or blank IDs
clean_df = clean_df.dropna(subset=['user_id', 'item_id', 'rating', 'timestamp'])
clean_df = clean_df[
    clean_df['user_id'].astype('string').str.strip().ne('')
    & clean_df['item_id'].astype('string').str.strip().ne('')
]

# Convert numeric columns and remove invalid values
clean_df['rating'] = pd.to_numeric(clean_df['rating'], errors='coerce')
clean_df['timestamp'] = pd.to_numeric(clean_df['timestamp'], errors='coerce')
clean_df = clean_df.dropna(subset=['rating', 'timestamp'])
clean_df = clean_df[clean_df['rating'].between(1, 5)]
clean_df = clean_df[clean_df['timestamp'] > 0]

# Sort before selecting one record for repeated user-item pairs
clean_df = clean_df.sort_values(['user_id', 'timestamp', 'item_id'])
clean_df = clean_df.drop_duplicates()
clean_df = clean_df.drop_duplicates(subset=['user_id', 'item_id'], keep='first')

print(f'Rows before cleaning: {before_rows:,}')
print(f'Rows after cleaning:  {len(clean_df):,}')
print(f'Rows removed:         {before_rows - len(clean_df):,}')
clean_df.head()

In [ ]:
# Convert timestamp to a readable datetime for inspection
clean_df['datetime'] = pd.to_datetime(clean_df['timestamp'], unit='s', errors='coerce')

# Final checks
print('Remaining null values:')
print(clean_df.isna().sum())
print('\nRating distribution:')
print(clean_df['rating'].value_counts().sort_index())
print('\nUnique users:', clean_df['user_id'].nunique())
print('Unique items:', clean_df['item_id'].nunique())
print('Date range:', clean_df['datetime'].min(), 'to', clean_df['datetime'].max())

# Sequence length per user
sequence_lengths = clean_df.groupby('user_id').size()
print('\nInteractions per user:')
print(sequence_lengths.describe())

In [ ]:
# Keep only users with at least five interactions for sequential modeling
MIN_INTERACTIONS_PER_USER = 5
eligible_users = sequence_lengths[sequence_lengths >= MIN_INTERACTIONS_PER_USER].index
model_df = clean_df[clean_df['user_id'].isin(eligible_users)].copy()

# Keep the model-ready columns and chronological order
model_df = model_df[['user_id', 'item_id', 'rating', 'timestamp', 'datetime']]
model_df = model_df.sort_values(['user_id', 'timestamp', 'item_id']).reset_index(drop=True)

print(f'Users retained: {model_df["user_id"].nunique():,}')
print(f'Items retained: {model_df["item_id"].nunique():,}')
print(f'Interactions retained: {len(model_df):,}')
model_df.head()

In [ ]:
# Save the cleaned data for the next modeling notebook
OUTPUT_FILE = 'amazon_beauty_cleaned.csv'
model_df.to_csv(OUTPUT_FILE, index=False)
print(f'Saved cleaned dataset to: {OUTPUT_FILE}')

## Cleaning summary

- Removed rows without a user, item, rating, or timestamp.
- Removed ratings outside the valid 1–5 range.
- Removed invalid timestamps.
- Removed exact duplicates.
- Kept one chronological interaction per repeated user-item pair.
- Removed users with fewer than five interactions.
- Saved the final chronological data as `amazon_beauty_cleaned.csv`.

For the fair baseline comparison, use this same cleaned file for centralized SASRec and FedSASRec. In FedSASRec, each `user_id` becomes one simulated client.